# Tunagem de modelo e criação de PKL
Aqui é aonde de fato utilizaremos o optuna com uma k folds do tipo times series split

## Import de bibliotecas

In [17]:
from utils.optuna_utils import objective
import pandas as pd
import matplotlib.pyplot as plt
import optuna
from lightgbm import LGBMClassifier
import pickle

## Tunagem

In [2]:
train_dataset = pd.read_csv('datasets/dataset_train.csv')

In [3]:
for col in train_dataset.select_dtypes(include=['object', 'string']).columns:
    train_dataset[col] = train_dataset[col].astype('category')
study = optuna.create_study(direction='maximize')
study.optimize(
    lambda trial: objective(
        trial=trial,
        random_state=42,
        X_train=train_dataset.drop(columns=['income']),
        y_train=train_dataset['income'],
        objective_metric='f1_score'
    ),
    n_trials=50
)

print("Melhores parâmetros encontrados:")
print(study.best_params)   
print(f"Melhor score médio: {study.best_value}")

[I 2026-05-22 20:21:22,982] A new study created in memory with name: no-name-07c3833b-81e9-40aa-a13f-0382b01bc378
[I 2026-05-22 20:22:28,087] Trial 0 finished with value: 0.6199789135817615 and parameters: {'boosting_type': 'dart', 'n_estimators': 1000, 'learning_rate': 0.0027833541868153744, 'max_depth': 3, 'num_leaves': 140, 'min_child_samples': 36, 'min_child_weight': 0.8444228209268615, 'min_split_gain': 0.9859228043887984, 'subsample': 0.6439481105395604, 'subsample_freq': 1, 'colsample_bytree': 0.6764587128306597, 'reg_alpha': 1.498884035038498e-05, 'reg_lambda': 1.4794737201256965e-07}. Best is trial 0 with value: 0.6199789135817615.
[I 2026-05-22 20:22:46,581] Trial 1 finished with value: 0.63622112794712 and parameters: {'boosting_type': 'gbdt', 'n_estimators': 600, 'learning_rate': 0.0011711458733633356, 'max_depth': 6, 'num_leaves': 57, 'min_child_samples': 30, 'min_child_weight': 0.007301652527016082, 'min_split_gain': 0.7424228374328894, 'subsample': 0.7644048867771132, 's

Melhores parâmetros encontrados:
{'boosting_type': 'gbdt', 'n_estimators': 900, 'learning_rate': 0.043390145553586085, 'max_depth': 6, 'num_leaves': 150, 'min_child_samples': 17, 'min_child_weight': 0.021769050136044802, 'min_split_gain': 0.17329792829363116, 'subsample': 0.8366401039209312, 'subsample_freq': 7, 'colsample_bytree': 0.9952954316420836, 'reg_alpha': 1.0023064257589987e-05, 'reg_lambda': 1.0894659767292119e-05}
Melhor score médio: 0.7769438377537574


## Extração de resultados e criação de modelo
Dada a tunagem, armazenamos os resultados e o modelo com melhores parâmetros

In [5]:
# Extrai os resultados para um DataFrame do Pandas
df_resultados = study.trials_dataframe()

# Ordena o DataFrame para mostrar os melhores resultados primeiro (maior score no topo)
df_resultados = df_resultados.sort_values(by='value', ascending=False)

# Visualiza as 5 melhores combinações
df_resultados.head()

,number,value,datetime_start,datetime_complete,duration,params_boosting_type,params_colsample_bytree,params_learning_rate,params_max_depth,params_min_child_samples,params_min_child_weight,params_min_split_gain,params_n_estimators,params_num_leaves,params_reg_alpha,params_reg_lambda,params_subsample,params_subsample_freq,state
42,42,0.776944,2026-05-22 21:06:48.892376,2026-05-22 21:07:00.047504,0 days 00:00:11.155128,gbdt,0.995295,0.043390,6,17,0.021769,0.173298,900,150,1.002306e-05,1.089466e-05,0.836640,7,COMPLETE
40,40,0.775761,2026-05-22 21:06:26.712792,2026-05-22 21:06:37.600838,0 days 00:00:10.888046,gbdt,0.999748,0.044255,6,18,0.025395,0.214470,900,119,3.161165e-05,5.115657e-07,0.825420,7,COMPLETE
43,43,0.774527,2026-05-22 21:07:00.051006,2026-05-22 21:07:08.099401,0 days 00:00:08.048395,gbdt,0.994291,0.068034,6,17,0.020908,0.181013,900,149,2.543278e-05,2.598147e-08,0.823977,7,COMPLETE
41,41,0.772413,2026-05-22 21:06:37.602833,2026-05-22 21:06:48.890380,0 days 00:00:11.287547,gbdt,0.975412,0.043636,6,19,0.020245,0.185942,1000,147,2.413409e-05,5.826043e-07,0.843586,7,COMPLETE
48,48,0.770577,2026-05-22 21:07:53.726609,2026-05-22 21:08:04.702984,0 days 00:00:10.976375,gbdt,0.999861,0.040856,7,34,0.007786,0.224109,800,145,5.154550e-07,2.040113e-06,0.765450,7,COMPLETE


In [6]:
df_resultados.to_csv('output_files/resultados_optuna.csv', index=False)

In [7]:
fixed_params = {
    'objective': 'multiclass',  # ou 'binary'
    'random_state': 42,
    'n_jobs': -1,
    'importance_type': 'split', 
    'class_weight': 'balanced',
    'verbose': -1
}

In [11]:
final_params = {**study.best_params, **fixed_params}
final_params

{'boosting_type': 'gbdt',
 'n_estimators': 900,
 'learning_rate': 0.043390145553586085,
 'max_depth': 6,
 'num_leaves': 150,
 'min_child_samples': 17,
 'min_child_weight': 0.021769050136044802,
 'min_split_gain': 0.17329792829363116,
 'subsample': 0.8366401039209312,
 'subsample_freq': 7,
 'colsample_bytree': 0.9952954316420836,
 'reg_alpha': 1.0023064257589987e-05,
 'reg_lambda': 1.0894659767292119e-05,
 'objective': 'multiclass',
 'random_state': 42,
 'n_jobs': -1,
 'importance_type': 'split',
 'class_weight': 'balanced',
 'verbose': -1}

In [12]:
X_final_train = train_dataset.drop(columns=['income'])
y_final_train = train_dataset['income']

In [15]:
final_model = LGBMClassifier(**final_params)
final_model.fit(X_final_train, y_final_train)

,boosting_type,'gbdt'
,num_leaves,150
,max_depth,6
,learning_rate,0.043390145553586085
,n_estimators,900
,subsample_for_bin,200000
,objective,'multiclass'
,class_weight,'balanced'
,min_split_gain,0.17329792829363116
,min_child_weight,0.021769050136044802
,min_child_samples,17


In [18]:
nome_arquivo_pickle = "lgbm_classifier_multiclass.pkl"

with open("output_files/" + nome_arquivo_pickle, "wb") as arquivo:
    pickle.dump(final_model, arquivo)